# Модуль линейной регрессии

In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv('data\\train.csv', index_col='id')

In [3]:
df_train

,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
id,,,,,,,,,,,,,,,,,,,
1,Aakash,Male,47,Agra,Working Professional,Teacher,NaN,1.0,NaN,NaN,5.0,Less than 5 hours,Unhealthy,B.Arch,Yes,11,5,Yes,0
2,Asha,Female,25,Chennai,Student,NaN,3.0,NaN,6.59,1.0,NaN,7-8 hours,Healthy,BSc,No,9,3,No,0
3,Samar,Male,56,Indore,Working Professional,Data Scientist,NaN,3.0,NaN,NaN,2.0,7-8 hours,Moderate,B.Tech,No,2,4,Yes,0
4,Chhavi,Female,24,Kalyan,Student,NaN,2.0,NaN,5.77,2.0,NaN,5-6 hours,Moderate,MBBS,Yes,5,3,No,1
5,Anand,Male,55,Kanpur,Working Professional,Researcher,NaN,1.0,NaN,NaN,2.0,7-8 hours,Unhealthy,BSc,Yes,4,4,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,Rahil,Male,50,Ahmedabad,Working Professional,Electrician,NaN,4.0,NaN,NaN,3.0,More than 8 hours,Healthy,BSc,Yes,2,5,Yes,0
1888,Rashi,Female,28,Vasai-Virar,Student,NaN,4.0,NaN,9.10,2.0,NaN,Less than 5 hours,Moderate,Class 12,Yes,11,5,Yes,1
1889,Barkha,Female,46,Kanpur,Working Professional,Content Writer,NaN,4.0,NaN,NaN,4.0,More than 8 hours,Unhealthy,BE,No,5,5,No,0


In [4]:
X = df_train.iloc[:, :-1]
y = df_train.iloc[:, -1]

Применяем преобразования к данным:

In [5]:
from sklearn.preprocessing import FunctionTransformer

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

In [6]:
X = prep.fit_transform(X)
X

,Name,Gender,Age,City,Working Professional or Student,Profession,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,job/study satisfaction,Work/Academic Pressure
id,,,,,,,,,,,,,,,
1,Aakash,Male,47,Agra,Working Professional,Teacher,Less than 5 hours,Unhealthy,B.Arch,Yes,11,5,Yes,5.0,1.0
2,Asha,Female,25,Chennai,Student,unemployed,7-8 hours,Healthy,BSc,No,9,3,No,1.0,3.0
3,Samar,Male,56,Indore,Working Professional,Data Scientist,7-8 hours,Moderate,B.Tech,No,2,4,Yes,2.0,3.0
4,Chhavi,Female,24,Kalyan,Student,unemployed,5-6 hours,Moderate,MBBS,Yes,5,3,No,2.0,2.0
5,Anand,Male,55,Kanpur,Working Professional,Researcher,7-8 hours,Unhealthy,BSc,Yes,4,4,No,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,Rahil,Male,50,Ahmedabad,Working Professional,Electrician,More than 8 hours,Healthy,BSc,Yes,2,5,Yes,3.0,4.0
1888,Rashi,Female,28,Vasai-Virar,Student,unemployed,Less than 5 hours,Moderate,Class 12,Yes,11,5,Yes,2.0,4.0
1889,Barkha,Female,46,Kanpur,Working Professional,Content Writer,More than 8 hours,Unhealthy,BE,No,5,5,No,4.0,4.0


In [7]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

Закодируем категориальные признаки OneHot энкодингом

In [8]:
cat_cols = X.select_dtypes(include=['str', 'category']).columns.tolist()

encoder = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ],
    remainder="passthrough"
).set_output(transform="pandas")

In [9]:
X_encoded = encoder.fit_transform(X)

In [10]:
X_encoded

,cat__Name_Aadhya,cat__Name_Aahana,cat__Name_Aakash,cat__Name_Aanchal,cat__Name_Aaradhya,cat__Name_Aarav,cat__Name_Aariv,cat__Name_Aarohi,cat__Name_Aarti,cat__Name_Aarush,...,cat__Degree_PhD,cat__Have you ever had suicidal thoughts ?_No,cat__Have you ever had suicidal thoughts ?_Yes,cat__Family History of Mental Illness_No,cat__Family History of Mental Illness_Yes,remainder__Age,remainder__Work/Study Hours,remainder__Financial Stress,remainder__job/study satisfaction,remainder__Work/Academic Pressure
id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,47,11,5,5.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,25,9,3,1.0,3.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,56,2,4,2.0,3.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,24,5,3,2.0,2.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,55,4,4,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,50,2,5,3.0,4.0
1888,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,28,11,5,2.0,4.0
1889,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,46,5,5,4.0,4.0


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

Построим пайплайн:

In [12]:
pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

In [13]:
X = df_train.iloc[:, :-1]

Функция кросс-валидации:

In [14]:
from sklearn.model_selection import cross_val_score

def cv(model, X, y):
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='f1'
    )
    
    print(scores.mean())

Оценка модели:

In [15]:
cv(pipe, X, y)

0.7888548596577794


In [16]:
pipe.fit(X, y)

,steps,"[('prep', ...), ('encoder', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function pre...0020A8BB9D6C0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [17]:
X_test = pd.read_csv("data\\test.csv", index_col='id') 

In [18]:
y_pred = pipe.predict(X_test)

In [19]:
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg.csv", index_label='id')

Попробуем дополнительно исключить некоторые признаки:

In [20]:
from sklearn.compose import make_column_selector

encoder = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
).set_output(transform="pandas")

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Name', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

cv(pipe, X, y)

0.8958207236094189


In [21]:
pipe.fit(X, y)
y_pred = pipe.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg2.csv", index_label='id')

Добавим label encoding

In [22]:
from sklearn.preprocessing import OrdinalEncoder

encoder = ColumnTransformer(
    transformers = [
        ("onehot", OneHotEncoder(sparse_output=False), ['City', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']),
        ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
    ],
    remainder="passthrough",
)

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

cv(pipe, X, y)

0.8974308567137556


In [23]:
pipe.fit(X, y)
y_pred = pipe.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg3.csv", index_label='id')

Сделаем подбор гиперпараметров:

In [24]:
from sklearn.model_selection import RandomizedSearchCV

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

param_dist = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "model__solver": ["lbfgs", "liblinear"],
    "model__penalty": ["l2"]
}

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.889543780305524


In [25]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg4.csv", index_label='id')

Исклюим еще признак City

In [26]:
def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.fillna({'Profession': 'unemployed'}, inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

encoder = ColumnTransformer(
    transformers = [
        ("onehot", OneHotEncoder(sparse_output=False), ['Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']),
        ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
    ],
    remainder="passthrough",
)

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.9021307024660989


In [27]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg5.csv", index_label='id')

Добавим полиномаильные признаки:

In [29]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

encoder = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False), make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)


0.9133051427773513


In [30]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg6.csv", index_label='id')

Попробуем исключить признак Profession

In [32]:
def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

encoder = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False), make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits', 'Degree']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.9435984786166506


In [33]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg7.csv", index_label='id')

Попробуем не удалять колонку CGPA, а заполнить пропуски средним:

In [48]:
from sklearn.impute import SimpleImputer

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)

    return X

prep = FunctionTransformer(preprocess, validate=False)

impute_ct = ColumnTransformer([
    ("impute_cgpa", SimpleImputer(strategy="mean"), ["CGPA"])
], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")

poly_ct = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False),
     make_column_selector(dtype_include=np.number))
], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")

encoder = ColumnTransformer([
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits', 'Degree']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("impute", impute_ct),
    ("poly", poly_ct),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)


0.5803192199633237


Качество упало, удаляем признак CGPA, попробуем исключить еще признак Degree:

In [49]:
def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

encoder = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False), make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.9605154977206155


In [50]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg8.csv", index_label='id')

Попробуем убрать стандартизацию:

In [63]:
def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)

    return X

prep = FunctionTransformer(preprocess, validate=False)

num = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num, make_column_selector(dtype_include=np.number)),
    ("onehot", OneHotEncoder(sparse_output=False), ['Sleep Duration', 'Dietary Habits']),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("preprocess", preprocessor),
    ("model", LogisticRegression()),
])

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.9594187677468307


In [53]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg9.csv", index_label='id')

Посмотрим на количество уникальных категорий признаков Sleep Duration и Dietary Habits

In [54]:
X['Sleep Duration'].unique()

<StringArray>
['Less than 5 hours', '7-8 hours', '5-6 hours', 'More than 8 hours']
Length: 4, dtype: str

In [55]:
X['Dietary Habits'].unique()

<StringArray>
['Unhealthy', 'Healthy', 'Moderate']
Length: 3, dtype: str

Их немного, поэтому попробуем перевести их в числовые значения вручную:

In [65]:
mapping_sleep = {
    'Less than 5 hours': 0,
    '5-6 hours': 1,
    '7-8 hours': 2,
    'More than 8 hours': 3,
}

mapping_diet = {
    'Unhealthy': 0,
    'Moderate': 1,
    'Healthy': 2,
}

def preprocess(X):
    X = X.copy()

    X['job/study satisfaction'] = X['Job Satisfaction'].fillna(X['Study Satisfaction'])
    X['Work/Academic Pressure'] = X['Academic Pressure'].fillna(X['Work Pressure'])
    X.drop(columns=['Degree', 'Profession', 'Name', 'City', 'Job Satisfaction', 'Study Satisfaction', 'Academic Pressure', 'Work Pressure'], inplace=True)
    X.drop('CGPA', inplace=True, axis=1)
    X['Sleep Duration'] = X['Sleep Duration'].map(mapping_sleep)
    X['Dietary Habits'] = X['Dietary Habits'].map(mapping_diet)

    return X

prep = FunctionTransformer(preprocess, validate=False)

encoder = ColumnTransformer([
    ("poly", PolynomialFeatures(degree=2, include_bias=False), make_column_selector(dtype_include=np.number)),
    ("label", OrdinalEncoder(), ['Gender', 'Working Professional or Student', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']),
])

pipe = Pipeline([
    ("prep", prep),
    ("encoder", encoder),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])

param_dist = {
    "model__C": np.logspace(-3, 3, 20),
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear", "saga"],
    "model__class_weight": [None, "balanced"]
}

model = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

cv(model, X, y)

0.9676462636960705


In [66]:
model.fit(X, y)
y_pred = model.predict(X_test)
df_out = pd.DataFrame(y_pred, columns=["Depression"])
df_out.index = df_out.index + 1
df_out.to_csv("out\\logreg10.csv", index_label='id')